In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Preprocess patient data

In [2]:
from tabular_study import clean_diagnose, one_hot_encode

In [ ]:
dataset_path = "/data/mvillota/Datasets/Datasets/Processed/Oslo_viability/original.xlsx"
df = pd.read_excel(dataset_path, sheet_name="Morula_Blast_nov")
df_patient = pd.read_excel(dataset_path, sheet_name="patients")
df_corr = pd.read_excel(dataset_path, sheet_name="behandling")
df_nul = pd.read_excel(dataset_path, sheet_name="Retrospektiv")

In [ ]:
len(df), len(df_patient), len(df_corr), len(df_nul)

(3387, 4690, 12160, 4171)

In [ ]:
df_patient.rename(columns={'Patient id fripro': 'Patient_id'}, inplace=True)
df.rename(columns={'OPU_ID_Number (corresponds to treatment id) fripro': 'OPU_ID_Number'}, inplace=True)
df_corr.rename(columns={'Patient Fripro id': 'Patient_id', 'OPU_ID_Number (corresponds to treatment id) fripro': 'OPU_ID_Number'}, inplace=True)
df_corr = df_corr[['Patient_id', 'OPU_ID_Number']]

In [ ]:
df1_patient = pd.merge(df_corr, df_patient, on='Patient_id', how='inner')
df1 = pd.merge(df, df_corr, on='OPU_ID_Number', how='inner')

In [93]:
df_merged = pd.merge(df1, df1_patient, on='OPU_ID_Number', how='left')

In [94]:
len(df_merged), len(df1), len(df1_patient)

(3350, 3350, 4690)

In [95]:
df = df_merged

In [64]:
implantation = {
    'mi': None,
    'hcg testpositiv': 1,
    'pos': 1,
    'positiv': 1,
    'biokjemisk': 1,       
    'neg': 0,
    'negativ': 0,
    'negative': 0,
    'negativ (står positiv i lf uten mer informasjon, men neg. i sys.x.)': 0
}
df['HCG Test'] = df['HCG Test'].map(implantation)
df.dropna(subset=["HCG Test"], inplace=True)
df = df.reset_index(drop=True)
df['Fetal Heart Beat']= df['Fetal Heart Beat'].fillna(0)
df.loc[df['Fetal Heart Beat'] == 'mi', 'Fetal Heart Beat'] = None

In [66]:
smoke_map = {
    # --- No smoker ---
    '0': 0,
    '0': 0,
    'nei': 0,
    
    # --- Smokers ---
    '3': 1,
    '5': 1,
    '0-5/d': 1,
    '1-5': 1,
    '1 (0-5/d)': 1,
    '1 (ikke daglig)': 1,

    # --- Heavy Smokers ---
    '1-10/d': 2,
    '10/d': 2,
    '1 (6-15/d)': 2,
    '1 (1-10/d)': 2,
    
    # --- noise ---
    'mi': None  # Se convertirá en NaN en Pandas
}
df['smoke'] = df['smoke'].map(smoke_map)

In [67]:
snus_map = {
    # --- No snus (0) ---
    '0': 0,
    'nei': 0,
    
    # --- snus (1) ---
    '1': 1,
    '2': 1,
    '3': 1,
    '5': 1,
    '4-5pk/d': 1,
    '1 (en sjelden gang)': 1,  
    '1 (0-1d/v)': 1,
    '1-2 d/v': 1,
    '1(slutter)': 1,           
    
    # --- Noise ---
    'mi': None                 
}
df['snus'] = df['snus'].map(snus_map)

In [68]:
df_merged['Wom_Smoker'].value_counts()

Wom_Smoker
Non smoker      60
Ikke røyker     55
Røyker           5
Smoker; n:o:     1
Name: count, dtype: int64

In [69]:
df[['Wom_Smoker', 'smoke']]

,Wom_Smoker,smoke
0,NaN,NaN
1,Non smoker,NaN
2,NaN,NaN
3,Non smoker,NaN
4,Non smoker,NaN
...,...,...
454,NaN,0.0
455,NaN,NaN
456,NaN,NaN
457,NaN,0.0


In [70]:
df.loc[df['Wom_Smoker']== 'Non smoker', 'smoke'] = 0
df.loc[df['Wom_Smoker']== 'Ikke røyker', 'smoke'] = 0
df.loc[df['Wom_Snuff']== 'Non snuffer', 'snus'] = 0
df.loc[df['Wom_Snuff']== 'Ikke snuser', 'snus'] = 0
df.loc[df['Wom_Snuff']== 'Snuff; n:o:', 'snus'] = 1

In [71]:
df['Wom_Smoker_Number_Per_Day'] = df['Wom_Smoker_Number_Per_Day'].map(smoke_map)
df.loc[df['Wom_Smoker']== 'Røyker', 'smoke'] = df.loc[df['Wom_Smoker']== 'Røyker', 'Wom_Smoker_Number_Per_Day']

In [72]:
df.drop(columns=['Wom_Smoker', 'Wom_Smoker_Number_Per_Day', 'Wom_Snuff', 'Wom_Snuff_Amount_Per_Week'], inplace=True)

In [73]:
df['Wom_BMI'].isna().sum()

np.int64(296)

In [74]:
df

,OPU_ID_Number,,.1,Embryo_id,Selection,HCG Test,DET/TET,number of oocytes,IVF/ICSI,Age,...,Patient_id_y,year of birth,Man_BMI,year of birth man,Man_Height,Man_Weight,Wom_Weight,Wom_Height,Wom_BMI,Wom_Prev_Stim
0,84452.0,372.1,x,1,Transfer,0.0,NaN,4,ivf,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,77144.0,373.4,x,4,Transfer,1.0,NaN,8,ivf,34.0,...,895408.0,1982.0,NaN,1975.0,NaN,NaN,71,172.0,24,0.0
2,79387.0,374.2,x,2,Transfer,0.0,NaN,6,ivf,33.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,77269.0,375.6,x,6,Transfer,0.0,NaN,11,icsi,33.0,...,886797.0,1982.0,NaN,1980.0,NaN,NaN,70,159.0,"27,7",0.0
4,76449.0,376.5,x,1,Transfer,1.0,NaN,16,ivf,29.0,...,888667.0,1986.0,NaN,1985.0,NaN,NaN,75,170.0,26,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
454,78760.0,832.1,x,1,Transfer,1.0,NaN,3,ivf,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
455,77004.0,833.2,x,2,Transfer,1.0,NaN,9,icsi,35.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
456,77506.0,834.2,x,2,Transfer,0.0,NaN,4,icsi,39.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
457,75875.0,835.1b,x,1,Transfer,0.0,"SET, but two videos",15,ivf,41.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
map_ivf = {'IVF': 0, 'ICSI': 1, 'ivf': 0, 'icsi': 1, 'ICSI-PESA': 1, 'ivf/icsi (icsi)': 1, 'ICSI - oocyttaktivering':1, 'IVF-nat':0, 'ivf/icsi (ivf)':0,'icsi-PESA': 1,
           'ivf/icsi': None, 'IVF/ICSI': None, }
df['IVF/ICSI'] = df['IVF/ICSI'].map(map_ivf)

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 90 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   OPU_ID_Number                     459 non-null    float64
 1                                     459 non-null    object 
 2    .1                               455 non-null    object 
 3   Embryo_id                         459 non-null    int64  
 4   Selection                         459 non-null    object 
 5   HCG Test                          459 non-null    float64
 6   DET/TET                           15 non-null     object 
 7   number of oocytes                 453 non-null    object 
 8   IVF/ICSI                          430 non-null    float64
 9   Age                               454 non-null    float64
 10  Age_mann                          454 non-null    float64
 11   .2                               1 non-null      object 
 12  Diagnose

In [77]:
df.columns

Index(['OPU_ID_Number', ' ', ' .1', 'Embryo_id', 'Selection', 'HCG Test',
       'DET/TET', 'number of oocytes', 'IVF/ICSI', 'Age', 'Age_mann', ' .2',
       'Diagnose', 'smoke', 'snus', ' .3', 'Abortion Comment',
       'Fetal Heart Beat', 'Gestational Sacs', 'sex', 'week', 'lenght',
       'weight', 'complications', 'End Time', 'Fertilization', 'Runs',
       'Slide Description', 'ID', 'Start Time', 'Transferred', 'Well', 'tPB2',
       'tPNa', 'tPNf', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9+', 'tSC',
       'tM', 'tSB', 'tB', 'tEB', 'tHB', 'Blastomere Size - Time',
       'Blastomere Size - Value', 'Fragmentation - Time',
       'Fragmentation - Value', 'Irregular Division - Time',
       'Irregular Division - Value', 'Multi Nucleation - Time',
       'Multi Nucleation - Value', 'Nuclei - Time', 'Nuclei - Value',
       'Pronuclei - Time', 'Pronuclei - Value', 'Inner Cell Mass - Time',
       'Inner Cell Mass - Value', 'Trophectoderm Evaluation - Time',
       'Trophectoderm E

In [78]:
df.drop(columns=['Slide Description',' ', ' .1',' .2',' .3',], inplace=True)

In [79]:
diagnose = [str(d).lower() for d in df["Diagnose"]]
list_diseases = clean_diagnose(diagnose)
df = one_hot_encode(df, list_diseases)

In [ ]:
df.loc[df['number of oocytes'] == '19 (6/13)', 'IVF/ICSI'] = 1
df.loc[df['number of oocytes'] == '19 (6/13)', 'number of oocytes'] = 13
df.loc[df['number of oocytes'] == '10 (6/4)', 'IVF/ICSI'] = 1
df.loc[df['number of oocytes'] == '10 (6/4)', 'number of oocytes'] = 4
df.loc[df['number of oocytes'] == '22 (16 ivf)', 'IVF/ICSI'] = 0
df.loc[df['number of oocytes'] == '22 (16 ivf)', 'number of oocytes'] = 16
df.loc[df['number of oocytes'] == '24 (12/12)', 'IVF/ICSI'] = 1
df.loc[df['number of oocytes'] == '24 (12/12)', 'number of oocytes'] = 12
df.loc[df['number of oocytes'] == '15 (8/7)', 'IVF/ICSI'] = 0
df.loc[df['number of oocytes'] == '15 (8/7)', 'number of oocytes'] = 12
df['number of oocytes'][370] = 4
df['number of oocytes'][371] = 6
df.loc[(df['IVF/ICSI'].isna()) & (df['Tse'] > 10), 'IVF/ICSI'] = 0
df.loc[(df['IVF/ICSI'].isna()) & (df['Tse'] < 10), 'IVF/ICSI'] = 0

In [148]:
#save the dataframe to a CSV file
output_path = "/data/mvillota/Datasets/Datasets/Processed/Oslo_viability/all_patient_morphk.csv"
df.to_csv(output_path, index=False)

# 2. Patient data

In [153]:
df = pd.read_csv(output_path)

In [154]:
df.columns

Index(['OPU_ID_Number', 'Embryo_id', 'Selection', 'HCG Test', 'DET/TET',
       'number of oocytes', 'IVF/ICSI', 'Age', 'Age_mann', 'Diagnose', 'smoke',
       'snus', 'Abortion Comment', 'Fetal Heart Beat', 'Gestational Sacs',
       'sex', 'week', 'lenght', 'weight', 'complications', 'End Time',
       'Fertilization', 'Runs', 'ID', 'Start Time', 'Transferred', 'Well',
       'tPB2', 'tPNa', 'tPNf', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9+',
       'tSC', 'tM', 'tSB', 'tB', 'tEB', 'tHB', 'Blastomere Size - Time',
       'Blastomere Size - Value', 'Fragmentation - Time',
       'Fragmentation - Value', 'Irregular Division - Time',
       'Irregular Division - Value', 'Multi Nucleation - Time',
       'Multi Nucleation - Value', 'Nuclei - Time', 'Nuclei - Value',
       'Pronuclei - Time', 'Pronuclei - Value', 'Inner Cell Mass - Time',
       'Inner Cell Mass - Value', 'Trophectoderm Evaluation - Time',
       'Trophectoderm Evaluation - Value', 'USRVAR_1_Multinucleation - Time',

In [155]:
df = df[['ID', 'Well', 'HCG Test', 'Fetal Heart Beat',
          'number of oocytes', 'IVF/ICSI', 
          'Age', 'Wom_Weight', 'Wom_Height', 'Wom_BMI', 'Wom_Prev_Stim',
          'smoke', 'snus', 
          'Age_mann', 'Man_Height', 'Man_Weight', 'Man_BMI',
          'ovulatory_factor', 'tubal_factor', 'endometriosis_factor',
          'male_factor', 'age_factor', 'uterine_factor', 'unexplained_factor'] ]

In [156]:
df.drop(columns=['Man_Height', 'Man_Weight', 'Man_BMI'], inplace=True)

In [158]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    459 non-null    object 
 1   Well                  459 non-null    int64  
 2   HCG Test              459 non-null    float64
 3   Fetal Heart Beat      456 non-null    float64
 4   number of oocytes     453 non-null    float64
 5   IVF/ICSI              459 non-null    float64
 6   Age                   454 non-null    float64
 7   Wom_Weight            163 non-null    float64
 8   Wom_Height            163 non-null    float64
 9   Wom_BMI               163 non-null    object 
 10  Wom_Prev_Stim         250 non-null    float64
 11  smoke                 323 non-null    float64
 12  snus                  321 non-null    float64
 13  Age_mann              454 non-null    float64
 14  ovulatory_factor      459 non-null    int64  
 15  tubal_factor          4